In [1]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import box
import sys
import numpy as np
import networkx as nx
import osmnx as ox
import matplotlib.pyplot as plt

In [2]:
sys.path.append('../src/nlr_ghe_screen/clustering/')
from distance_matrices import (
    construct_euclidian_dist_matrix,
    construct_road_network_dist_matrix,
)
from seed_clustering import (
    seed_clusters, 
    find_seed_centroids,
)
from construct_clusters import (
    build_clusters,
    create_cluster_subgraph
)

sys.path.append('../src/nlr_ghe_screen/api/')
from building_footprint_api import (
    get_overture_footprints,
    get_ornl_footprints,
    join_ornl_overture_buildings
    
)
from osmnx_api import (
    get_road_network,
    plot_network_graph
)

In [3]:
TEST_BBOX = (47.58, -122.36, 47.61, -122.31) # xmin, ymin, xmax, ymax
south, west, north, east = TEST_BBOX

In [4]:
ghe_locations = gpd.read_file("../src/exports/ghe_locations.geojson")
geom_box = box(west, south, east, north)
cropped_ghes = gpd.clip(ghe_locations, geom_box)
cropped_ghes = cropped_ghes.to_crs(cropped_ghes.estimate_utm_crs())

In [5]:
overture_buildings = get_overture_footprints(bbox=TEST_BBOX)
ornl_buildings = get_ornl_footprints(bbox=TEST_BBOX)


---------------------------
Retrieving Overture Maps Data
Success - Data retrieved
---------------------------

---------------------------
Retrieving ORNL Structure Data
total features: 2679
2000/2679
2679/2679
Success - Data retrieved
---------------------------


In [6]:
ornl_buildings = ornl_buildings.to_crs(cropped_ghes.crs)
overture_buildings = overture_buildings.to_crs(cropped_ghes.crs)

In [7]:
network = get_road_network(bbox=TEST_BBOX)


---------------------------
Post - Networkx road network request
Success - Query Received
---------------------------


In [8]:
network = network.to_undirected()
network.is_directed()

False

In [9]:
dist_matrix = construct_road_network_dist_matrix(
    gdf=cropped_ghes,
    G=network,
    max_distance=1000,
)

In [10]:
buildings_joined = join_ornl_overture_buildings(
    ornl_data=ornl_buildings,
    overture_data=overture_buildings
)


In [12]:
cluster_labels = seed_clusters(
    dist_matrix=dist_matrix,
)
cropped_ghes['cluster'] = cluster_labels

In [13]:
seed_centroids = find_seed_centroids(
    ghe_locations=cropped_ghes,
    cluster_col='cluster'
)

In [18]:
des_clusters = build_clusters(
    seed_centroids=seed_centroids,
    building_data=buildings_joined,
    G=network,
    max_distance=500,
)

In [19]:
c1 = cropped_ghes['cluster'].value_counts().rename("n1")
c2 = des_clusters['cluster'].value_counts().rename("n2")

counts = pd.concat([c1, c2], axis=1).fillna(0).astype(int)
counts["min"] = counts[["n1", "n2"]].min(axis=1)

ranked = counts.sort_values("min", ascending=False)      
ordered_labels = ranked.index.tolist()
ordered_labels[:10]

[1, 0, 4, 2, 5, 9, 17, 7, 3, 21]

In [66]:
idx = 2

In [67]:
mst_subgraph = create_cluster_subgraph(
    cluster_id=idx,
    building_data=des_clusters,
    ghe_locations=cropped_ghes,
    G=network
)

In [68]:
from shapely.geometry import LineString, Point
import folium

def plot_network_graph(
    network: nx.MultiDiGraph, 
    extra_layers: list
):
    """
    """
    edge_lines = []
    for u, v, data in network.edges(data=True):
        if 'geometry' in data:
            geom = data['geometry']
        else:
            # Build a LineString from node coordinates
            x1, y1 = network.nodes[u]['x'], network.nodes[u]['y']
            x2, y2 = network.nodes[v]['x'], network.nodes[v]['y']
            geom = LineString([(x1, y1), (x2, y2)])
        edge_lines.append({'u': u, 'v': v, 'geometry': geom})

    gdf_edges = gpd.GeoDataFrame(edge_lines, crs=network.graph['crs'])

    # Convert NetworkX nodes to a GeoDataFrame of Points
    node_points = []
    for n, data in network.nodes(data=True):
        node_points.append(
            {'node': n, 'geometry': Point(data['x'], data['y']), **data}
        )

    gdf_nodes = gpd.GeoDataFrame(node_points, crs=network.graph['crs'])

    # 3. Create the base interactive map using GeoPandas .explore()
    # m = base_gdf.explore(color='green', name='Base Map')

    m = gdf_edges.explore(color='blue', weight=2, name='Network Edges')
    m = gdf_nodes.explore(m=m, color='red', marker_kwds={'radius': 4}, name='Network Nodes')

    for name, gdf, color in (extra_layers or []):
        m = gdf.explore(m=m, color=color, name=name)

    # Fit to everything combined, since each explore() call re-fits to its own data
    all_bounds = [gdf_edges.to_crs(4326).total_bounds]
    all_bounds += [g.to_crs(4326).total_bounds for _, g, _ in (extra_layers or [])]
    minx = min(b[0] for b in all_bounds); miny = min(b[1] for b in all_bounds)
    maxx = max(b[2] for b in all_bounds); maxy = max(b[3] for b in all_bounds)
    m.fit_bounds([[miny, minx], [maxy, maxx]])

    folium.LayerControl().add_to(m)
    return m

In [69]:
m = plot_network_graph(
    mst_subgraph,
    extra_layers=[
        ('GHEs', cropped_ghes[cropped_ghes['cluster'] == idx], 'green'),
        ('Buildings', des_clusters[des_clusters['cluster'] == idx], 'blue')
    ],
)
m

In [70]:
cropped_ghes.explore()